# Parallel 05 — ZeRO / FSDP：把显存墙按 N 切开

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Papa-Panda/field-notes/blob/main/pytorch/parallel/05-zero-fsdp.ipynb)

- **日期**：2026-10-02
- **定位**：并行基础线第二篇，接 Day 01 留下的尾巴 —— DDP 把同步做便宜了，但每个 rank 还存着全量参数 + 全量优化器状态，1.1TB 的账一分没少。ZeRO 拆的就是这笔账。
- **环境**：纯 PyTorch，CPU + gloo 即可验证分片机制。

配套笔记：[05-zero-fsdp.md](05-zero-fsdp.md)


## 0. 因果链定位

DDP 的账：16 字节/参数（bf16 参数 2 + bf16 梯度 2 + fp32 主参数 4 + Adam 动量 4 + 方差 4），每个 rank 全额存。$ N $ 个 rank 里躺着 $ N $ 份一模一样的优化器状态，而每一步之后，每个 rank 只需要更新属于自己的那一片参数。

ZeRO 的洞察一句话：**计算可以复制，状态不必复制。** 把重复存储的东西依次切片：

- ZeRO-1 切优化器状态（12B 的大头）—— 不参与通信，免费午餐
- ZeRO-2 再切梯度（all-reduce 换 reduce-scatter，通信不变）
- ZeRO-3 再切参数（用时 all-gather 拼回，前向+反向各一次 = 多付 50% 通信）

FSDP = ZeRO-3 的 PyTorch 原生实现。今天算清账、跑通分片、验证每 rank 真的只存一半。


## 1. 先算账：三级分片各省多少

$ \Psi $ = 参数量，$ N $ = rank 数。显存是每 rank 的字节数（乘以 $ \Psi $ ），通信按 DDP 的 $ 2\Psi $ 为基准。


In [ ]:
def zero_table(Psi_B, N):
    """Psi_B: 参数量（十亿）。打印各级 ZeRO 每 rank 显存（GB）与通信倍数。"""
    rows = [
        ('DDP',    16,              2),
        ('ZeRO-1', 4 + 12 / N,     2),
        ('ZeRO-2', 2 + 14 / N,     2),
        ('ZeRO-3', 16 / N,         3),
    ]
    print(f'{"方案":8s} {"每参数字节":>10s} {"每 rank 显存":>12s} {"通信":>6s}')
    for name, bpp, comm in rows:
        print(f'{name:8s} {bpp:8.2f} B {bpp * Psi_B:10.1f} GB {comm / 2:.1f}x')

zero_table(70, 8)
print('\n读法：70B 模型 8 卡：DDP 每卡 1120GB（不可能），ZeRO-3 每卡 140GB（还是大，但进入可讨论范围）。')
print('N 拉到 64：ZeRO-3 每卡 17.5GB —— 这就是为什么大模型训练动辄几百上千卡。')


## 2. FSDP 的实现：flatten、all-gather、reshard

FSDP 把每个被 wrap 的模块的参数**拍平成一维** `FlatParameter` 再按 rank 切片。每 rank 只持有自己那片；前向需要某模块的参数时，先 all-gather 拼出全量、算完立刻扔掉（reshard）；反向同理。

两个工程细节，面试常问：

- **prefetch**：算当前层时提前 all-gather 下一层的参数，把通信藏进计算里 —— 和 DDP 的"通信与 backward 重叠"是同一种思想（拿调度换总时间）。
- **`summon_full_params`**：保存 checkpoint 或做评估时，在这个上下文里临时拼出全量参数。忘了它直接拿 `state_dict()`，存下来的是半份模型 —— 本节代码里会当场对比。


## 3. 实跑：2 进程 FSDP，验证"每 rank 只存一半"

CPU + gloo 就能验证分片机制（慢是慢，账是同一本账）。


In [ ]:
import os, torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

class MLP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(64, 256), torch.nn.ReLU(),
            torch.nn.Linear(256, 256), torch.nn.ReLU(),
            torch.nn.Linear(256, 10))
    def forward(self, x):
        return self.net(x)

def run(rank, world_size):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = '29503'
    dist.init_process_group('gloo', rank=rank, world_size=world_size)
    torch.manual_seed(0)

    base = MLP()
    total = sum(p.numel() for p in base.parameters())
    model = FSDP(base, device_id=torch.device('cpu'))  # CPU 验证分片机制；GPU 上换成当前 cuda 设备
    local = sum(p.numel() for p in model.parameters())

    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    x = torch.randn(32, 64); y = torch.randint(0, 10, (32,))
    losses = []
    for _ in range(10):
        opt.zero_grad()
        loss = torch.nn.functional.cross_entropy(model(x), y)
        loss.backward(); opt.step()
        losses.append(loss.item())

    # 拼全量验证：summon 后参数应与 total 一致且跨 rank 相同
    with FSDP.summon_full_params(model):
        full = sum(p.numel() for p in model.parameters())
        p0 = torch.cat([p.data.view(-1) for p in model.parameters()])
    gathered = [torch.zeros_like(p0) for _ in range(world_size)]
    dist.all_gather(gathered, p0)

    if rank == 0:
        print(f'全量参数: {total:,} | 每 rank 实际持有: {local:,}（≈ 全量/{world_size}）')
        print(f'summon 后参数量: {full:,}（应等于全量）| 跨 rank 最大差: {(gathered[0]-gathered[1]).abs().max().item():.2e}')
        print(f'loss: {losses[0]:.3f} -> {losses[-1]:.3f}（在降 = 分片没把训练弄坏）')
    dist.destroy_process_group()

if __name__ == '__main__':
    mp.spawn(run, args=(2,), nprocs=2, join=True)


## 4. 调试清单

- **存了半份模型**：`state_dict()` 直接存的是分片。保存前用 FSDP 的 full state dict 模式或 `summon_full_params` 拼全量。症状是 load 时 shape 对不上。
- **wrap 粒度**：整个模型包成一个 FlatParameter → 一次 all-gather 拼全量，峰值显存爆炸；按层包太细 → all-gather 次数爆炸、延迟主导。按 transformer block 包是常规解。
- **DDP 思维乱入**：FSDP 不用 `find_unused_parameters` 那套；有参数没参与计算时行为与 DDP 不同，先查模块有没有真的被调用。
- **手动摸分片参数**：在外面拿 `param.shape` 当全量形状用 → 各种诡异 shape 报错。摸参数前先想想它现在是不是分片状态。

## 小结

- 省显存的顺序就是重复程度的顺序：优化器状态（纯重复）→ 梯度（可规约分发）→ 参数（用时拼回）。
- ZeRO-3 的交易条款：显存降到 $ 16\Psi/N $ ，通信涨到 $ 3\Psi $ （1.5 倍）。值不值取决于瓶颈是显存还是带宽 —— 大模型永远是显存先死，所以这笔交易基本必做。
- FSDP 的手感三件套：flatten 分片、按需 all-gather、`summon_full_params` 拼全量。

**下一篇**：Parallel 06 — TP 手写 linear：把一层网络横着切开（RL track 电话点名的题目）。
